In [ ]:
!mkdir dataset
%cd dataset

# download images
!wget http://images.cocodataset.org/zips/train2017.zip
!unzip train2017.zip

# download annotations
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip annotations_trainval2017.zip

Streaming output truncated to the last 5000 lines.
 extracting: train2017/000000249290.jpg  
 extracting: train2017/000000025529.jpg  
 extracting: train2017/000000316928.jpg  
 extracting: train2017/000000337866.jpg  
 extracting: train2017/000000547768.jpg  
 extracting: train2017/000000423162.jpg  
 extracting: train2017/000000224149.jpg  
 extracting: train2017/000000117841.jpg  
 extracting: train2017/000000251660.jpg  
 extracting: train2017/000000110997.jpg  
 extracting: train2017/000000424728.jpg  
 extracting: train2017/000000384745.jpg  
 extracting: train2017/000000475535.jpg  
 extracting: train2017/000000252604.jpg  
 extracting: train2017/000000002525.jpg  
 extracting: train2017/000000547307.jpg  
 extracting: train2017/000000546568.jpg  
 extracting: train2017/000000002024.jpg  
 extracting: train2017/000000162559.jpg  
 extracting: train2017/000000395397.jpg  
 extracting: train2017/000000255134.jpg  
 extracting: train2017/000000549188.jpg  
 extracting: train2017/00

In [ ]:
import json              # import json module to read/write JSON files
import os                # import os module (not used here but for file handling if needed)

# load annotations
with open('annotations/instances_train2017.json') as f:   # open the COCO annotation file
    data = json.load(f)                                   # load JSON content into Python dictionary

# keep only person and chair
allowed_classes = [1, 62]   # category IDs (1 = person, 62 = chair)

filtered_images = []        # list to store filtered image info
filtered_annotations = []   # list to store filtered annotations

image_ids = set()           # set to store unique image IDs (no duplicates)

for ann in data['annotations']:                 # loop through all annotations
    if ann['category_id'] in allowed_classes:   # check if annotation is person/chair
        filtered_annotations.append(ann)        # keep this annotation
        image_ids.add(ann['image_id'])          # store its image ID

for img in data['images']:              # loop through all images
    if img['id'] in image_ids:          # check if image has relevant annotation
        filtered_images.append(img)     # keep this image

filtered_data = {
    "images": filtered_images,   # filtered images list
    "annotations": filtered_annotations,  # filtered annotations list
    "categories": [cat for cat in data['categories'] if cat['id'] in allowed_classes]  # keep only required categories
}

with open('filtered.json', 'w') as f:   # create new JSON file
    json.dump(filtered_data, f)         # write filtered data into file

print("Filtered dataset created!")      # confirmation message

Filtered dataset created!


Convert COCO Annotations to YOLO Label Format (Person & Chair)

In [ ]:
import os                          # import os module for directory/file handling

os.makedirs("labels", exist_ok=True)   # create 'labels' folder if it doesn't exist

#  Create fast lookup dictionary (VERY IMPORTANT)
image_dict = {img['id']: img for img in filtered_images}
# create dictionary: image_id → image_info (faster lookup than looping)

def convert_bbox(size, box):      # function to convert COCO bbox → YOLO format
    dw = 1. / size[0]             # normalize width (1 / image width)
    dh = 1. / size[1]             # normalize height (1 / image height)
    x = box[0] + box[2]/2         # convert top-left x → center x
    y = box[1] + box[3]/2         # convert top-left y → center y
    w = box[2]                    # width of bounding box
    h = box[3]                    # height of bounding box
    return (x*dw, y*dh, w*dw, h*dh)   # return normalized (x, y, w, h)

for ann in filtered_annotations:      # loop through all filtered annotations
    img_id = ann['image_id']          # get image ID for this annotation

    #  FAST lookup instead of slow search
    img_info = image_dict[img_id]     # get image info using dictionary (O(1) lookup)

    file_name = img_info['file_name'] # image file name (e.g., 000000123.jpg)
    width = img_info['width']         # image width
    height = img_info['height']       # image height

    bbox = ann['bbox']                # COCO bbox format [x, y, width, height]
    x, y, w, h = convert_bbox((width, height), bbox)  # convert to YOLO format

    class_id = 0 if ann['category_id'] == 1 else 1
    # assign class: 0 = person, 1 = chair

    label_path = f"labels/{file_name.replace('.jpg','.txt')}"
    # create label file path (same name as image but .txt)

    with open(label_path, "a") as f:   # open file in append mode
        f.write(f"{class_id} {x} {y} {w} {h}\n")
        # write YOLO format: class x_center y_center width height

print("YOLO labels created!")         # confirmation message

YOLO labels created!


Convert COCO to YOLO Labels with Progress Tracking & Safety Check

In [ ]:
import os                          # import os module for file/directory operations

os.makedirs("labels", exist_ok=True)   # create 'labels' folder if it doesn't exist

# Fast lookup dictionary
image_dict = {img['id']: img for img in filtered_images}
# map image_id → image_info for quick access

def convert_bbox(size, box):      # function to convert COCO bbox → YOLO format
    dw = 1. / size[0]             # normalize width (1 / image width)
    dh = 1. / size[1]             # normalize height (1 / image height)
    x = box[0] + box[2]/2         # convert top-left x → center x
    y = box[1] + box[3]/2         # convert top-left y → center y
    w = box[2]                    # bounding box width
    h = box[3]                    # bounding box height
    return (x*dw, y*dh, w*dw, h*dh)   # return normalized YOLO format

total = len(filtered_annotations)   # total number of annotations
print(f"Total annotations to process: {total}")   # print total count

for i, ann in enumerate(filtered_annotations):   # loop with index (for progress tracking)
    img_id = ann['image_id']          # get image ID

    # Safety check (avoid crash)
    if img_id not in image_dict:      # skip if image not found
        continue

    img_info = image_dict[img_id]     # get image details

    file_name = img_info['file_name'] # image file name
    width = img_info['width']         # image width
    height = img_info['height']       # image height

    bbox = ann['bbox']                # COCO bbox [x, y, w, h]
    x, y, w, h = convert_bbox((width, height), bbox)  # convert to YOLO format

    class_id = 0 if ann['category_id'] == 1 else 1
    # assign class: 0 = person, 1 = chair

    label_path = f"labels/{file_name.replace('.jpg','.txt')}"
    # create label file path

    with open(label_path, "a") as f:   # open label file in append mode
        f.write(f"{class_id} {x} {y} {w} {h}\n")
        # write annotation in YOLO format

    # 🔥 Progress print every 1000 steps
    if i % 1000 == 0:
        print(f"Processed {i}/{total} annotations ({(i/total)*100:.2f}%)")
        # show progress percentage

print(" YOLO labels created successfully!")   # final success message

Total annotations to process: 300956
Processed 0/300956 annotations (0.00%)
Processed 1000/300956 annotations (0.33%)
Processed 2000/300956 annotations (0.66%)
Processed 3000/300956 annotations (1.00%)
Processed 4000/300956 annotations (1.33%)
Processed 5000/300956 annotations (1.66%)
Processed 6000/300956 annotations (1.99%)
Processed 7000/300956 annotations (2.33%)
Processed 8000/300956 annotations (2.66%)
Processed 9000/300956 annotations (2.99%)
Processed 10000/300956 annotations (3.32%)
Processed 11000/300956 annotations (3.66%)
Processed 12000/300956 annotations (3.99%)
Processed 13000/300956 annotations (4.32%)
Processed 14000/300956 annotations (4.65%)
Processed 15000/300956 annotations (4.98%)
Processed 16000/300956 annotations (5.32%)
Processed 17000/300956 annotations (5.65%)
Processed 18000/300956 annotations (5.98%)
Processed 19000/300956 annotations (6.31%)
Processed 20000/300956 annotations (6.65%)
Processed 21000/300956 annotations (6.98%)
Processed 22000/300956 annotat

Create Smaller Dataset (Random Sampling of Images & Labels)

In [ ]:
import shutil        # import shutil for copying files
import random        # import random for sampling files

os.makedirs("images_small", exist_ok=True)   # create folder for sampled images
os.makedirs("labels_small", exist_ok=True)   # create folder for sampled labels

files = list(os.listdir("labels"))   # get list of all label files
selected = random.sample(files, 600)   # randomly pick 600 label files

for file in selected:                     # loop through selected label files
    img = file.replace(".txt", ".jpg")   # convert label filename → image filename

    shutil.copy(f"train2017/{img}", f"images_small/{img}")
    # copy corresponding image to small dataset folder

    shutil.copy(f"labels/{file}", f"labels_small/{file}")
    # copy label file to small dataset folder

print("Reduced dataset ready!")   # confirmation message

Reduced dataset ready!


In [ ]:
!zip -r dataset.zip images_small labels_small

  adding: images_small/ (stored 0%)
  adding: images_small/000000054277.jpg (deflated 0%)
  adding: images_small/000000401054.jpg (deflated 0%)
  adding: images_small/000000010966.jpg (deflated 0%)
  adding: images_small/000000515198.jpg (deflated 1%)
  adding: images_small/000000019441.jpg (deflated 0%)
  adding: images_small/000000247712.jpg (deflated 0%)
  adding: images_small/000000147793.jpg (deflated 0%)
  adding: images_small/000000568309.jpg (deflated 0%)
  adding: images_small/000000273841.jpg (deflated 0%)
  adding: images_small/000000330400.jpg (deflated 0%)
  adding: images_small/000000081593.jpg (deflated 0%)
  adding: images_small/000000495525.jpg (deflated 2%)
  adding: images_small/000000497591.jpg (deflated 0%)
  adding: images_small/000000566026.jpg (deflated 0%)
  adding: images_small/000000441275.jpg (deflated 0%)
  adding: images_small/000000102763.jpg (deflated 0%)
  adding: images_small/000000330420.jpg (deflated 0%)
  adding: images_small/000000450728.jpg (defla

In [ ]:
from google.colab import files
files.download("dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
person_files = []
chair_files = []

for file in os.listdir("labels"):
    with open(f"labels/{file}") as f:
        content = f.read()

        if content.startswith("0"):  # person
            person_files.append(file)
        elif content.startswith("1"):  # chair
            chair_files.append(file)

print("Person:", len(person_files))
print("Chair:", len(chair_files))

Person: 59753
Chair: 9101


Create Balanced Dataset (Equal Person & Chair Samples)

In [ ]:
import random        # import random for sampling
import shutil        # import shutil for copying files
import os            # import os for directory handling

os.makedirs("images_balanced", exist_ok=True)   # create folder for balanced images
os.makedirs("labels_balanced", exist_ok=True)   # create folder for balanced labels

# Take equal samples
num = min(len(person_files), len(chair_files), 300)
# choose smallest count among both classes (max 300)

person_sample = random.sample(person_files, num)   # randomly pick person label files
chair_sample = random.sample(chair_files, num)     # randomly pick chair label files

selected = person_sample + chair_sample   # combine both samples

for file in selected:                     # loop through selected files
    img = file.replace(".txt", ".jpg")   # convert label filename → image filename

    shutil.copy(f"train2017/{img}", f"images_balanced/{img}")
    # copy corresponding image

    shutil.copy(f"labels/{file}", f"labels_balanced/{file}")
    # copy corresponding label

print("Balanced dataset created!")   # confirmation message

Balanced dataset created!


In [ ]:
!zip -r dataset_balanced.zip images_balanced labels_balanced

  adding: images_balanced/ (stored 0%)
  adding: images_balanced/000000541293.jpg (deflated 0%)
  adding: images_balanced/000000398200.jpg (deflated 0%)
  adding: images_balanced/000000435908.jpg (deflated 0%)
  adding: images_balanced/000000545072.jpg (deflated 0%)
  adding: images_balanced/000000386694.jpg (deflated 0%)
  adding: images_balanced/000000504917.jpg (deflated 3%)
  adding: images_balanced/000000051258.jpg (deflated 0%)
  adding: images_balanced/000000345218.jpg (deflated 0%)
  adding: images_balanced/000000168521.jpg (deflated 1%)
  adding: images_balanced/000000281040.jpg (deflated 0%)
  adding: images_balanced/000000251367.jpg (deflated 0%)
  adding: images_balanced/000000039747.jpg (deflated 0%)
  adding: images_balanced/000000380539.jpg (deflated 0%)
  adding: images_balanced/000000437117.jpg (deflated 0%)
  adding: images_balanced/000000440763.jpg (deflated 0%)
  adding: images_balanced/000000229960.jpg (deflated 0%)
  adding: images_balanced/000000365822.jpg (defla

In [ ]:
from google.colab import files
files.download("dataset_balanced.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("Images:", len(os.listdir("images_balanced")))
print("Labels:", len(os.listdir("labels_balanced")))

Images: 600
Labels: 600
